# Hybrid Labeling Pipeline: Open Buildings Map & LiDAR Z-Confidence Filter

**Project:** Integrating High-Resolution Imagery and Machine Learning for Small-Area Population Estimation in Melusi (Atteridgeville, Pretoria)  
**Pipeline owner:** Kampamba Chanda (U-Net segmentation)  
**Interface consumer:** Lehlogonolo Mbiza (Random Forest regression)

---

## Design rationale

This notebook implements a hybrid labeling strategy for generating U-Net training masks:

1. **Open Buildings v3** provides 15,047 satellite-derived footprint polygons as the base mask source.
2. **LiDAR Z-distribution** acts as a confidence filter — footprints with insufficient height above local ground (< 1.5 m) are suppressed as likely false positives (walls, fences, concrete pads).
3. The 10th percentile of Z values within each footprint polygon serves as a local ground proxy, eliminating the need for full CSF ground classification.

### Confidence tiers
| Tier | Height above ground | Interpretation |
|------|-------------------|----------------|
| `high` | ≥ 2.5 m | Unambiguous structure |
| `medium` | 1.5 – 2.5 m | Plausible dwelling |
| `suppressed` | < 1.5 m | Likely false positive — excluded from masks |
| `no_lidar` | Insufficient points | Retained but flagged |

### Key outputs
- Binary mask GeoTIFFs per block (aligned to Stage A rasters)
- 512 × 512 mask tiles (aligned to Stage B image tiles)
- `block_dwelling_counts.csv`, `tile_index.csv`, `footprint_confidence.gpkg`

### Known constraints
- Block rasters are stored in **EPSG:4148** (Hartebeesthoek94 geographic); footprints are reprojected from Lo29 to match before rasterization.
- Tile slicing writes to local SSD first, then bulk-copies to Google Drive to avoid FUSE I/O throttling.
- Open Buildings was trained on ~50 cm satellite imagery, small informal dwellings (< 3–4 m footprint) may be missed at this resolution.

### Expected runtime (Colab L4 Runtime)
| Step | Time |
|------|------|
| Phase 0 | ~34 sec |
| Phase 1 | ~5 sec |
| Phase 2 | ~2820 sec |
| Phase 3 | ~14 sec |
| Phase 4 | ~1080 sec |
| Phase X | ~6 sec |
| **Total** | ~66 mins |


# Phase 0: Set up (Runtime = 34s)

In [35]:
# ============ BOOTSTRAP ============
import subprocess, sys, os
from google.colab import drive, userdata

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Load credentials from Colab Secrets
TOKEN = userdata.get('gitToken')
NAME  = userdata.get('gitName')
EMAIL = userdata.get('gitMail')

# 3. Git identity
!git config --global user.name "{NAME}"
!git config --global user.email "{EMAIL}"

# 4. Clone repo (skip if already cloned)
REPO_DIR = '/content/mit808-2026-project-data-insight-drivers'
if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/up-mitc-ds/mit808-2026-project-data-insight-drivers.git
else:
    print("Repo already cloned")

# 5. Switch to branch and sync with master
os.chdir(REPO_DIR)
!git fetch origin master
!git merge origin/master -m "Sync with master"
!git checkout -b kc/label_mask

# 6. Install dependencies
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    os.path.join(REPO_DIR, 'requirements.txt')
])

# 7. Make src/ importable
sys.path.insert(0, REPO_DIR)
from src.setup import configure_environment, PATHS, save_and_push
configure_environment()

print("Bootstrap complete")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned
From https://github.com/up-mitc-ds/mit808-2026-project-data-insight-drivers
 * branch            master     -> FETCH_HEAD
Already up to date.
fatal: A branch named 'kc/label_mask' already exists.
✅ All 10 required packages available
✅ Plot defaults configured
✅ Output directories verified
────────────────────────────────────────
🚀 Environment ready
   Data:    /content/drive/MyDrive/MIT/MIT808/melusi-2025-data/raw/
   Repo:    /content/mit808-2026-project-data-insight-drivers
   Figures: /content/mit808-2026-project-data-insight-drivers/reports/figures
Bootstrap complete


# Phase 1: Prepare Open Buildings footprints (Runtime = 5s)

Manually download the Open Buildings v3 CSV for the S2 cell covering Melusi from
[sites.research.google/open-buildings](https://sites.research.google/open-buildings/),

Melusi Boundary box in WGS84: [ 28.09723346 -25.72851691  28.12802482 -25.71700787] from Boandary notebook:


In [2]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import rasterio

OB_CSV = "/content/drive/MyDrive/MIT/MIT808/melusi-2025-data/external/melusi_ob.csv.gz"
OB_GPKG = "/content/open_buildings.gpkg"
RASTER_REF = "/content/drive/MyDrive/MIT/MIT808/melusi-2025-data/raw/Melusi_Georeferenced/Melusi_2025.tif"

if not os.path.exists(OB_GPKG):
    df = pd.read_csv(OB_CSV)
    df["geometry"] = df["geometry"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

    # Reproject to match raster CRS (EPSG:4148)
    with rasterio.open(RASTER_REF) as src:
        raster_crs = src.crs
        print(f"Raster CRS: {raster_crs}")

    gdf = gdf.to_crs(raster_crs)
    gdf.to_file(OB_GPKG, driver="GPKG")
    print(f"Saved {len(gdf)} footprints to {OB_GPKG}")
else:
    print(f"Open Buildings GeoPackage already exists at {OB_GPKG}")


Raster CRS: EPSG:4148
Saved 15047 footprints to /content/open_buildings.gpkg


# Phase 2: Hybrid labeling pipeline (Runtime = 47m)


In [4]:
import os, re, json, gc, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import laspy
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely import contains_xy
from scipy.spatial import cKDTree
from pathlib import Path
from tqdm import tqdm


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    DRIVE_BASE = "/content/drive/MyDrive/MIT/MIT808/melusi-2025-data"
    RAW_DIR = f"{DRIVE_BASE}/raw"
    PROCESSED_DIR = f"{DRIVE_BASE}/processed"

    BLOCK_RASTERS_DIR = f"{PROCESSED_DIR}/smoothBlocks/block_rasters"
    STAGE_B_TILES_DIR = f"{PROCESSED_DIR}/smoothBlocks/melusi_tiles"
    MASKS_DIR = f"{PROCESSED_DIR}/masks"
    MASK_TILES_DIR = f"{PROCESSED_DIR}/smoothBlocks/melusi_mask_tiles"

    LIDAR_LOCAL = "/content/local_data/"
    LIDAR_SRC = f"{RAW_DIR}/Point_cloud/Melusi_pointcloud_07032025_lo29.las"
    BLOCK_GPKG = f"{PROCESSED_DIR}/smoothBlocks/zone_boundaries_lo29.gpkg"
    OPEN_BUILDINGS_PATH = "/content/open_buildings.gpkg"

    LOCAL_MASK_TILES = "/content/local_masks"

    CRS_LO29 = "EPSG:2053"
    CRS_WGS84 = "EPSG:4326"

    TILE_SIZE = 512
    PIXEL_SIZE = 0.03
    HEIGHT_THRESHOLD_LOW = 1.5
    HEIGHT_THRESHOLD_HIGH = 2.5
    GROUND_PERCENTILE = 10
    MIN_POINTS_IN_FOOTPRINT = 5


# ============================================================================
# STEP 0: COPY LIDAR TO LOCAL STORAGE
# ============================================================================

def copy_lidar_local(cfg):
    os.makedirs(cfg.LIDAR_LOCAL, exist_ok=True)
    dst = os.path.join(cfg.LIDAR_LOCAL, os.path.basename(cfg.LIDAR_SRC))
    if os.path.exists(dst):
        print("LiDAR already on local SSD, skipping copy.")
    else:
        print("Copying LiDAR to local storage...")
        shutil.copy(cfg.LIDAR_SRC, dst)
        print("  Done.")


# ============================================================================
# STEP 1: LOAD OPEN BUILDINGS + SPATIAL JOIN
# ============================================================================

def load_and_join(cfg):
    print("Step 1: Loading Open Buildings + spatial join to blocks")

    gdf = gpd.read_file(cfg.OPEN_BUILDINGS_PATH)
    if gdf.crs is None:
        gdf = gdf.set_crs(cfg.CRS_WGS84)
    gdf = gdf.to_crs(cfg.CRS_LO29)

    if "footprint_id" not in gdf.columns:
        gdf["footprint_id"] = [f"fp_{i:05d}" for i in range(len(gdf))]

    print(f"  {len(gdf)} footprints loaded")

    blocks = gpd.read_file(cfg.BLOCK_GPKG)
    if "Blocks" in blocks.columns:
        blocks["Block_id"] = blocks["Blocks"].apply(lambda x: f"block_{int(x)}")
    elif "Block_id" not in blocks.columns:
        blocks["Block_id"] = [f"block_{i}" for i in range(len(blocks))]

    joined = gpd.sjoin(gdf, blocks[["Block_id", "geometry"]], how="inner")
    joined = joined.drop(columns=["index_right"], errors="ignore")
    print(f"  {len(joined)} matched to {joined['Block_id'].nunique()} blocks")
    return joined


# ============================================================================
# STEP 2: LIDAR Z-STATISTICS (BLOCK-WISE, MEMORY-SAFE)
# ============================================================================

def compute_z_stats(footprints, cfg):
    print("Step 2: Computing per-footprint Z statistics")

    las_path = str(next(
        p for ext in ("*.las", "*.LAS")
        for p in Path(cfg.LIDAR_LOCAL).glob(ext)
    ))
    results = []

    for block_id, group in tqdm(footprints.groupby("Block_id"), desc="Z-stats"):
        minx, miny, maxx, maxy = group.total_bounds
        buf = 5.0
        minx -= buf; miny -= buf; maxx += buf; maxy += buf

        las = laspy.read(las_path)
        bbox_mask = (
            (las.x >= minx) & (las.x <= maxx) &
            (las.y >= miny) & (las.y <= maxy)
        )

        if bbox_mask.sum() == 0:
            for _, row in group.iterrows():
                results.append(_empty_z_record(row["footprint_id"], 0))
            del las; gc.collect()
            continue

        xyz = np.column_stack([las.x[bbox_mask], las.y[bbox_mask], las.z[bbox_mask]])
        del las; gc.collect()

        tree = cKDTree(xyz[:, :2])

        for _, row in group.iterrows():
            geom = row.geometry
            bounds = geom.bounds
            cx, cy = geom.centroid.x, geom.centroid.y
            dx, dy = bounds[2] - bounds[0], bounds[3] - bounds[1]
            radius = np.sqrt(dx**2 + dy**2) / 2 + 1.0

            idx = tree.query_ball_point([cx, cy], radius)

            if len(idx) < cfg.MIN_POINTS_IN_FOOTPRINT:
                results.append(_empty_z_record(row["footprint_id"], len(idx)))
                continue

            cands = xyz[idx]
            inside = cands[contains_xy(geom, cands[:, 0], cands[:, 1])]

            if len(inside) < cfg.MIN_POINTS_IN_FOOTPRINT:
                results.append(_empty_z_record(row["footprint_id"], len(inside)))
                continue

            z = inside[:, 2]
            z_p10 = np.percentile(z, cfg.GROUND_PERCENTILE)
            results.append({
                "footprint_id": row["footprint_id"],
                "n_points": len(inside),
                "z_p10": z_p10,
                "z_max": z.max(),
                "z_median": np.median(z),
                "height_above_ground": z.max() - z_p10,
            })

        del xyz, tree; gc.collect()

    stats_df = pd.DataFrame(results)
    footprints = footprints.merge(stats_df, on="footprint_id", how="left")
    print(f"  Z-stats computed for {stats_df['z_max'].notna().sum()} / {len(stats_df)} footprints")
    return footprints


def _empty_z_record(fp_id, n):
    return {"footprint_id": fp_id, "n_points": n,
            "z_p10": np.nan, "z_max": np.nan,
            "z_median": np.nan, "height_above_ground": np.nan}


# ============================================================================
# STEP 3: CONFIDENCE CLASSIFICATION
# ============================================================================

def classify_confidence(footprints, cfg):
    print("Step 3: Classifying confidence levels")

    conditions = [
        footprints["height_above_ground"].isna(),
        footprints["height_above_ground"] < cfg.HEIGHT_THRESHOLD_LOW,
        footprints["height_above_ground"] < cfg.HEIGHT_THRESHOLD_HIGH,
        footprints["height_above_ground"] >= cfg.HEIGHT_THRESHOLD_HIGH,
    ]
    footprints["confidence"] = np.select(
        conditions, ["no_lidar", "suppressed", "medium", "high"], default="unknown"
    )

    counts = footprints["confidence"].value_counts()
    for label in ["high", "medium", "suppressed", "no_lidar"]:
        print(f"  {label:>12s}: {counts.get(label, 0):>6,}")
    return footprints


# ============================================================================
# STEP 4: RASTERIZE MASKS (CRS-AWARE)
# ============================================================================

def rasterize_masks(footprints, cfg):
    """
    Rasterize validated footprints per block.

    Critical: block rasters are EPSG:4148 while footprints are EPSG:2053.
    We reproject footprints to match the raster CRS before burning pixels.
    """
    print("Step 4: Rasterizing masks per block")
    os.makedirs(cfg.MASKS_DIR, exist_ok=True)

    valid = footprints[footprints["confidence"].isin(["high", "medium", "no_lidar"])].copy()
    mask_paths = {}

    for block_id, group in tqdm(valid.groupby("Block_id"), desc="Rasterizing"):
        block_tiff = os.path.join(cfg.BLOCK_RASTERS_DIR, f"{block_id}.tif")
        if not os.path.exists(block_tiff):
            continue

        with rasterio.open(block_tiff) as src:
            raster_crs = src.crs
            transform = src.transform
            h, w = src.height, src.width

        # Reproject footprints to raster CRS (EPSG:4148)
        group_reproj = group.to_crs(raster_crs)

        mask = rasterize(
            [(geom, 1) for geom in group_reproj.geometry],
            out_shape=(h, w),
            transform=transform,
            fill=0, dtype="uint8",
            all_touched=True,
        )

        mask_path = os.path.join(cfg.MASKS_DIR, f"{block_id}_mask.tif")
        with rasterio.open(
            mask_path, "w", driver="GTiff",
            height=h, width=w, count=1, dtype="uint8",
            crs=raster_crs, transform=transform, compress="deflate",
        ) as dst:
            dst.write(mask, 1)

        mask_paths[block_id] = mask_path
        px = int(mask.sum())
        print(f"  {block_id}: {px:,} dwelling px ({100 * px / (h * w):.2f}%)")

    print(f"  Masks for {len(mask_paths)} / 12 blocks")
    return mask_paths


# ============================================================================
# STEP 5: TILE INDEX + MASK SLICING (LOCAL SSD → DRIVE BULK COPY)
# ============================================================================

def build_tile_index(cfg):
    """Reconstruct tile_index from existing Stage B tiles via inverse affine."""
    print("Step 5a: Building tile index from Stage B tiles")

    tile_files = sorted(Path(cfg.STAGE_B_TILES_DIR).glob("tile_*_block_*.tif"))
    print(f"  {len(tile_files)} Stage B tiles found")

    pattern = re.compile(r"tile_(\d+)_block_(\d+)\.tif")

    # Cache block transforms (12 reads, not 8K)
    block_transforms = {}
    for bt in Path(cfg.BLOCK_RASTERS_DIR).glob("block_*.tif"):
        with rasterio.open(str(bt)) as src:
            block_transforms[bt.stem] = src.transform

    records = []
    for tf in tqdm(tile_files, desc="Indexing"):
        m = pattern.match(tf.name)
        if not m:
            continue
        block_id = f"block_{int(m.group(2))}"
        if block_id not in block_transforms:
            continue

        with rasterio.open(str(tf)) as src:
            tx, ty = src.transform.c, src.transform.f

        inv = ~block_transforms[block_id]
        col_off, row_off = inv * (tx, ty)

        records.append({
            "tile_id": f"tile_{int(m.group(1))}_block_{int(m.group(2))}",
            "Block_id": block_id,
            "tile_path": str(tf),
            "row_off": int(round(row_off)),
            "col_off": int(round(col_off)),
        })

    tile_df = pd.DataFrame(records)
    print(f"  Indexed {len(tile_df)} tiles across {tile_df['Block_id'].nunique()} blocks")
    return tile_df


def slice_mask_tiles(tile_df, mask_paths, cfg):
    """
    Slice block masks into 512×512 tiles on local SSD, then bulk-copy to Drive.
    Writing to local disk is ~10–20× faster than direct Drive FUSE writes.
    """
    print("Step 5b: Slicing mask tiles (local SSD → Drive)")
    os.makedirs(cfg.LOCAL_MASK_TILES, exist_ok=True)
    os.makedirs(cfg.MASK_TILES_DIR, exist_ok=True)

    open_masks = {bid: rasterio.open(mp) for bid, mp in mask_paths.items()}

    mask_tile_paths, dwelling_counts = [], []
    try:
        for _, row in tqdm(tile_df.iterrows(), total=len(tile_df), desc="Slicing"):
            bid = row["Block_id"]
            if bid not in open_masks:
                mask_tile_paths.append(None); dwelling_counts.append(0)
                continue

            src = open_masks[bid]
            win = Window(int(row["col_off"]), int(row["row_off"]), cfg.TILE_SIZE, cfg.TILE_SIZE)
            tile = src.read(1, window=win, boundless=True, fill_value=0)
            t_transform = rasterio.windows.transform(win, src.transform)

            fname = f"{row['tile_id']}_mask.tif"
            local_path = os.path.join(cfg.LOCAL_MASK_TILES, fname)

            with rasterio.open(
                local_path, "w", driver="GTiff",
                height=cfg.TILE_SIZE, width=cfg.TILE_SIZE,
                count=1, dtype="uint8",
                crs=src.crs, transform=t_transform, compress="deflate",
            ) as dst:
                dst.write(tile, 1)

            mask_tile_paths.append(os.path.join(cfg.MASK_TILES_DIR, fname))
            dwelling_counts.append(int(tile.sum()))
    finally:
        for s in open_masks.values():
            s.close()

    # Bulk copy to Drive
    print("  Copying to Google Drive...")
    local_files = os.listdir(cfg.LOCAL_MASK_TILES)
    for f in tqdm(local_files, desc="Drive copy"):
        src_p = os.path.join(cfg.LOCAL_MASK_TILES, f)
        dst_p = os.path.join(cfg.MASK_TILES_DIR, f)
        if not os.path.exists(dst_p):
            shutil.copy2(src_p, dst_p)

    tile_df["mask_path"] = mask_tile_paths
    tile_df["has_mask"] = tile_df["mask_path"].notna()
    tile_df["dwelling_pixel_count"] = dwelling_counts
    tile_df["dwelling_fraction"] = tile_df["dwelling_pixel_count"] / (cfg.TILE_SIZE ** 2)
    return tile_df


# ============================================================================
# STEP 6: INTERFACE FILES
# ============================================================================

def generate_interface_files(footprints, tile_df, cfg):
    print("Step 6: Generating interface files")

    valid = footprints[footprints["confidence"].isin(["high", "medium", "no_lidar"])]

    # Dwelling counts per block
    dc = (
        valid.groupby("Block_id")
        .agg(
            dwelling_count=("footprint_id", "count"),
            high_confidence=("confidence", lambda x: (x == "high").sum()),
            medium_confidence=("confidence", lambda x: (x == "medium").sum()),
            no_lidar=("confidence", lambda x: (x == "no_lidar").sum()),
            mean_height=("height_above_ground", "mean"),
            median_height=("height_above_ground", "median"),
        )
        .reset_index()
    )
    dc.to_csv(os.path.join(cfg.PROCESSED_DIR, "block_dwelling_counts.csv"), index=False)

    # Tile index
    tile_df.to_csv(os.path.join(cfg.PROCESSED_DIR, "tile_index.csv"), index=False)

    # Manifest
    manifest = {
        "pipeline": "hybrid_open_buildings_lidar", "version": "3.0",
        "parameters": {
            "height_threshold_low_m": cfg.HEIGHT_THRESHOLD_LOW,
            "height_threshold_high_m": cfg.HEIGHT_THRESHOLD_HIGH,
            "ground_percentile": cfg.GROUND_PERCENTILE,
            "tile_size_px": cfg.TILE_SIZE, "pixel_size_m": cfg.PIXEL_SIZE,
        },
        "summary": {
            "total_footprints": len(footprints),
            "validated": len(valid),
            "suppressed": int((footprints["confidence"] == "suppressed").sum()),
            "blocks": int(tile_df["Block_id"].nunique()),
            "tiles": len(tile_df),
            "tiles_with_dwellings": int((tile_df["dwelling_fraction"] > 0).sum()),
            "mean_dwelling_fraction": round(float(tile_df["dwelling_fraction"].mean()), 6),
        },
    }
    with open(os.path.join(cfg.PROCESSED_DIR, "block_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    # Footprint confidence GeoPackage
    footprints.to_file(os.path.join(cfg.PROCESSED_DIR, "footprint_confidence.gpkg"), driver="GPKG")

    print("  Saved: block_dwelling_counts.csv, tile_index.csv, block_manifest.json, footprint_confidence.gpkg")


In [5]:
cfg = Config()

copy_lidar_local(cfg)
footprints = load_and_join(cfg)
footprints = compute_z_stats(footprints, cfg)
footprints = classify_confidence(footprints, cfg)
mask_paths = rasterize_masks(footprints, cfg)
tile_df = build_tile_index(cfg)
tile_df = slice_mask_tiles(tile_df, mask_paths, cfg)
generate_interface_files(footprints, tile_df, cfg)

# Summary
total = len(tile_df)
with_dw = (tile_df["dwelling_fraction"] > 0).sum()
dense = (tile_df["dwelling_fraction"] > 0.1).sum()

print(f"\nTotal tiles:          {total:>8,}")
print(f"Tiles with dwellings: {with_dw:>8,}  ({100*with_dw/total:.1f}%)")
print(f"Dense tiles (>10%):   {dense:>8,}  ({100*dense/total:.1f}%)")
print(f"Mean dwelling frac:   {tile_df['dwelling_fraction'].mean():.6f}")

print(f"\nPer-block breakdown:")
for bid, grp in tile_df.groupby("Block_id"):
    n, dw = len(grp), (grp["dwelling_fraction"] > 0).sum()
    px = grp["dwelling_pixel_count"].sum()
    print(f"  {bid:>10s}: {n:>5,} tiles, {dw:>5,} with dwellings, {px:>10,} dwelling px")


Copying LiDAR to local storage...
  Done.
Step 1: Loading Open Buildings + spatial join to blocks
  15047 footprints loaded
  15028 matched to 12 blocks
Step 2: Computing per-footprint Z statistics


Z-stats: 100%|██████████| 12/12 [06:00<00:00, 30.05s/it]


  Z-stats computed for 13377 / 15028 footprints
Step 3: Classifying confidence levels
          high:  1,080
        medium:  4,040
    suppressed:  8,343
      no_lidar:  1,653
Step 4: Rasterizing masks per block


Rasterizing:   8%|▊         | 1/12 [00:02<00:22,  2.03s/it]

  block_1: 1,047,241 dwelling px (5.57%)


...

  Indexed 8109 tiles across 12 blocks
Step 5b: Slicing mask tiles (local SSD → Drive)


Slicing: 100%|██████████| 8109/8109 [03:23<00:00, 39.84it/s]


  Copying to Google Drive...


Drive copy: 100%|██████████| 8109/8109 [00:02<00:00, 2777.98it/s]


Step 6: Generating interface files
  Saved: block_dwelling_counts.csv, tile_index.csv, block_manifest.json, footprint_confidence.gpkg

Total tiles:             8,109
Tiles with dwellings:    5,454  (67.3%)
Dense tiles (>10%):      1,966  (24.2%)
Mean dwelling frac:   0.063236

Per-block breakdown:
     block_1:    74 tiles,    73 with dwellings,  1,686,551 dwelling px
    block_10: 1,116 tiles,   821 with dwellings, 19,614,110 dwelling px
    block_11:   629 tiles,   459 with dwellings, 14,227,185 dwelling px
    block_12: 1,619 tiles,   885 with dwellings, 30,984,790 dwelling px
     block_2:   408 tiles,   270 with dwellings,  5,915,581 dwelling px
     block_3:   877 tiles,   634 with dwellings,  9,982,868 dwelling px
     block_4:   870 tiles,   254 with dwellings,  5,130,626 dwelling px
     block_5:   402 tiles,   324 with dwellings,  6,302,204 dwelling px
     block_6:   782 tiles,   574 with dwellings, 15,627,700 dwelling px
     block_7:   422 tiles,   410 with dwellings, 10,6

# Phase 3: Visual validation (Runtime = 14s)

Centre-crop overlay of three blocks (sparse, dense, largest) to confirm masks
land on actual rooftops.


In [6]:
import matplotlib.pyplot as plt

for block_id in ["block_5", "block_7", "block_12"]:
    block_tiff = f"{cfg.BLOCK_RASTERS_DIR}/{block_id}.tif"
    mask_tiff = f"{cfg.MASKS_DIR}/{block_id}_mask.tif"

    with rasterio.open(block_tiff) as src:
        cy, cx = src.height // 2, src.width // 2
        win = Window(cx - 1024, cy - 1024, 2048, 2048)
        rgb = np.moveaxis(src.read([1, 2, 3], window=win), 0, -1)

    with rasterio.open(mask_tiff) as src:
        mask = src.read(1, window=win)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(rgb); axes[0].set_title(f"{block_id} — RGB")
    axes[1].imshow(mask, cmap="Reds"); axes[1].set_title("Mask")
    axes[2].imshow(rgb); axes[2].imshow(mask, cmap="Reds", alpha=0.4)
    axes[2].set_title("Overlay")
    for ax in axes: ax.axis("off")
    plt.suptitle(block_id, fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()


Output hidden; open in https://colab.research.google.com to view.

# Phase 4: Manual validation — 15% stratified sample  (Runtime = 18m)

Generates overlay PNGs for 15% of dwelling-containing tiles per block.
Each image title shows the footprint polygon count for that tile.

**Workflow:** Open the PNGs in Google Drive, count missed rooftops per image,
record in `validation_log.csv`.  
**Error metric:** `sum(missed_roofs) / sum(structures_detected) × 100`

**Confidence metric:** `(1 - Error) × 100`


In [7]:
from shapely.geometry import box

VALIDATION_DIR = f"{cfg.PROCESSED_DIR}/validation_overlays"
SAMPLE_FRACTION = 0.15
RANDOM_SEED = 42

os.makedirs(VALIDATION_DIR, exist_ok=True)

# Load validated footprints in raster CRS for spatial querying
fp_valid = gpd.read_file(os.path.join(cfg.PROCESSED_DIR, "footprint_confidence.gpkg"))
fp_valid = fp_valid[fp_valid["confidence"].isin(["high", "medium", "no_lidar"])].copy()

with rasterio.open(os.path.join(cfg.BLOCK_RASTERS_DIR, "block_1.tif")) as src:
    raster_crs = src.crs
fp_reproj = fp_valid.to_crs(raster_crs)
fp_reproj.sindex  # build spatial index

# Sample 15% of tiles with dwellings per block
dw_tiles = tile_df[tile_df["dwelling_fraction"] > 0].copy()
sampled = (
    dw_tiles.groupby("Block_id", group_keys=False)
    .apply(lambda g: g.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED))
)
print(f"Sampled {len(sampled)} tiles for validation")

# Cache open rasters
open_blocks = {}
open_masks_v = {}

def get_src(block_id, kind):
    cache = open_blocks if kind == "rgb" else open_masks_v
    if block_id not in cache:
        if kind == "rgb":
            cache[block_id] = rasterio.open(f"{cfg.BLOCK_RASTERS_DIR}/{block_id}.tif")
        else:
            cache[block_id] = rasterio.open(f"{cfg.MASKS_DIR}/{block_id}_mask.tif")
    return cache[block_id]

overlay_records = []
try:
    for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc="Overlays"):
        bid, tid = row["Block_id"], row["tile_id"]
        co, ro = int(row["col_off"]), int(row["row_off"])

        bsrc = get_src(bid, "rgb")
        msrc = get_src(bid, "mask")
        win = Window(co, ro, cfg.TILE_SIZE, cfg.TILE_SIZE)

        rgb = np.moveaxis(bsrc.read([1, 2, 3], window=win, boundless=True, fill_value=0), 0, -1)
        msk = msrc.read(1, window=win, boundless=True, fill_value=0)

        # Count footprint polygons in tile
        tt = rasterio.windows.transform(win, bsrc.transform)
        left, top = tt.c, tt.f
        right = left + cfg.TILE_SIZE * tt.a
        bottom = top + cfg.TILE_SIZE * tt.e
        minx, maxx = min(left, right), max(left, right)
        miny, maxy = min(top, bottom), max(top, bottom)
        n_fp = len(fp_reproj.sindex.query(box(minx, miny, maxx, maxy), predicate="intersects"))

        # Save overlay
        os.makedirs(os.path.join(VALIDATION_DIR, bid), exist_ok=True)
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(rgb)
        ax.imshow(msk, cmap="Reds", alpha=0.35, vmin=0, vmax=1)
        ax.set_title(f"{tid}  |  {n_fp} structures detected", fontsize=10, fontweight="bold")
        ax.axis("off"); plt.tight_layout()
        out = os.path.join(VALIDATION_DIR, bid, f"{tid}_overlay.png")
        fig.savefig(out, dpi=150, bbox_inches="tight", pad_inches=0.1)
        plt.close(fig)

        overlay_records.append({
            "tile_id": tid, "Block_id": bid,
            "structures_detected": n_fp,
            "dwelling_pixel_count": row["dwelling_pixel_count"],
            "overlay_path": out,
        })
finally:
    for s in list(open_blocks.values()) + list(open_masks_v.values()):
        s.close()

# Save validation log
val_df = pd.DataFrame(overlay_records)
val_df["missed_roofs"] = ""
val_df["notes"] = ""
val_df.to_csv(os.path.join(VALIDATION_DIR, "validation_log.csv"), index=False)

print(f"\n{len(val_df)} overlays saved to {VALIDATION_DIR}")
print(f"Estimated review time: ~{len(val_df) * 0.5:.0f} min at 30 sec/tile")


/tmp/ipykernel_11779/3623265939.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED))


Sampled 818 tiles for validation


Overlays: 100%|██████████| 818/818 [18:38<00:00,  1.37s/it]



818 overlays saved to /content/drive/MyDrive/MIT/MIT808/melusi-2025-data/processed/validation_overlays
Estimated review time: ~409 min at 30 sec/tile


# Phase x: Commit and push (Runtime = 6s)

In [9]:
import subprocess, os
from google.colab import userdata

# --- Secrets ---
token = userdata.get('gitToken')
name  = userdata.get('gitName')
mail  = userdata.get('gitMail')

repo_url = f"https://{token}@github.com/up-mitc-ds/mit808-2026-project-data-insight-drivers.git"
repo_dir = "/content/repo"

BRANCH_1 = "kc/tiling_pipeline"   # <-- replace if different

def run(cmd, cwd=repo_dir):
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
    else:
        print(result.stdout)
    return result

# --- Clone if not already present ---
if not os.path.exists(repo_dir):
    run(f"git clone {repo_url} {repo_dir}", cwd="/content")

run(f'git config user.name "{name}"')
run(f'git config user.email "{mail}"')
run(f"git remote set-url origin {repo_url}")

# --- Commit & push branch ---
run(f"git checkout {BRANCH_1}")
run("git add -A")
run(f'git commit -m "feat: finalise tiling pipeline - Estimated Runtime Corrected"')
run(f"git push origin {BRANCH_1}")

# --- Merge both into master ---
run("git checkout master")
run("git pull origin master")  # sync first

run(f"git merge --no-ff {BRANCH_1} -m 'merge: {BRANCH_1} into master'")

run("git push origin master")
print("Done — both branches merged into master.")




Your branch is up to date with 'origin/kc/tiling_pipeline'.


STDERR: 

Your branch is up to date with 'origin/master'.

Already up to date.

Already up to date.


Done — both branches merged into master.


In [42]:
save_and_push("kc_boundaries.ipynb", "Fixed haphazard boundary allocations v2")

data		   Makefile	  README.md	    setup.py
data_statement.md  model_card.md  references	    src
docs		   models	  reports	    test_environment.py
LICENSE		   notebooks	  requirements.txt  tox.ini
